# 03 · Lists and tuples

After this notebook you can choose between a list, a tuple and a deque with reasons, state the cost of every common list operation, copy nested data without aliasing bugs, and manipulate a chat-message history the way LLM apps do.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Variables, types and numbers](01_variables_types_numbers.ipynb)

## Why this matters in interviews

- Lists are the default container, so interviewers use them to test complexity (why is `insert(0, x)` slow?), aliasing (shallow vs deep copy) and idioms (`sorted` with `key`, unpacking).
- "List vs tuple" is a near-certain warm-up. The good answer is about *meaning* (a sequence vs a record) and hashability, not just "tuples are immutable".
- In LLM apps the conversation *is* a list of dicts. Trimming history, finding the last user turn and copying a shared system prompt safely are daily tasks — and a shallow-copy bug can leak one user's conversation state into another's.

## What interviewers ask

- What is the difference between a list and a tuple, and when do you use each?
- `sort()` vs `sorted()`? Is Python's sort stable? How do you sort by several keys?
- Shallow vs deep copy — give an example where a shallow copy causes a bug.
- What does `[[0] * 3] * 3` produce, and why is it a trap?
- Time complexity of `append`, `pop()`, `pop(0)`, `insert(0, x)`, `x in lst` and indexing. How can `append` be O(1)?
- How do you implement a queue efficiently in Python?
- Can a tuple ever change?

## 1 · Creating, indexing and slicing

A list is an ordered, **mutable** sequence of references to objects of any type. Indexing and slicing work exactly as for strings ([02](02_strings_and_fstrings.ipynb)); the difference is that you can also *assign* to an index or a slice — and a slice assignment may even change the length.

In [ ]:
tokens = ["The", "cat", "sat", "on", "the", "mat"]
print("tokens[0], tokens[-1]:", tokens[0], tokens[-1])
print("tokens[1:3]          :", tokens[1:3])
print("tokens[::-1]         :", tokens[::-1])
print("list('abc')          :", list("abc"), "| list(range(4)):", list(range(4)), "| [0] * 3:", [0] * 3)

tokens[0] = "A"                           # assign to an index
tokens[1:3] = ["dog", "ran", "off"]       # replace 2 items with 3
print("after assignments    :", tokens)
del tokens[-1]
print("after del tokens[-1] :", tokens)
assert tokens == ["A", "dog", "ran", "off", "on", "the"]

## 2 · Methods that change the list — and return `None`

| Method | Effect | Cost |
|---|---|---|
| `append(x)` | add one item at the end | O(1) amortised |
| `extend(iterable)` | add every item of an iterable at the end | O(k) |
| `insert(i, x)` | insert before index `i` | O(n) — shifts everything after `i` |
| `pop()` / `pop(i)` | remove and **return** the last / the `i`-th item | O(1) / O(n) |
| `remove(x)` | remove the **first** item equal to `x`; `ValueError` if absent | O(n) |
| `clear()` | empty the list (the same object) | O(n) |
| `index(x)`, `count(x)` | search | O(n) |
| `sort()`, `reverse()` | reorder in place | O(n log n), O(n) |

The in-place methods all return `None`, so `lst = lst.append(x)` replaces your list with `None`.

**Predict, then run:** what does `history` look like after `append` vs `extend` with a list argument? What is `result`?

In [ ]:
history = ["hi"]
history.append(["how", "are"])       # adds ONE item: a nested list
history.extend(["you", "?"])         # adds each item
print("history:", history)

history.insert(0, "<start>")
last = history.pop()
history.remove("hi")
print("after insert/pop/remove:", history, "| popped:", repr(last))

result = history.append("x")
print("result of append:", result)

alias = history
history.clear()                      # empties the SAME object; `history = []` would rebind instead
print("alias after clear():", alias)
try:
    history.remove("missing")
except ValueError as err:
    print("ValueError:", err)
assert result is None and alias == [] and alias is history

## 3 · `sort()` vs `sorted()`, keys and stability

- `lst.sort()` sorts **in place** and returns `None`; lists only.
- `sorted(iterable)` returns a **new** list from any iterable — a tuple, dict keys, a generator.
- `key=` is a function computed once per item and compared instead of the item (`key=str.lower`, `key=lambda c: c["score"]`, or `operator.itemgetter("score")`); `reverse=True` flips the order.
- Python's sort (Timsort) is **stable**: items with equal keys keep their original relative order. That lets you sort by several keys — in one pass with a tuple key, or in several passes, least important key first.

**Predict, then run:** `c2` and `c3` tie on score. Which comes first?

In [ ]:
chunks = [
    {"id": "c1", "doc": "faq",    "score": 0.72},
    {"id": "c2", "doc": "policy", "score": 0.91},
    {"id": "c3", "doc": "faq",    "score": 0.91},
    {"id": "c4", "doc": "blog",   "score": 0.40},
]
by_score = sorted(chunks, key=lambda c: c["score"], reverse=True)
print("by score desc     :", [c["id"] for c in by_score])

by_doc_then_score = sorted(chunks, key=lambda c: (c["doc"], -c["score"]))     # one pass, tuple key
print("by doc, then score:", [(c["doc"], c["id"]) for c in by_doc_then_score])

words = ["banana", "apple", "Cherry"]
print("sorted(words)     :", sorted(words), "<- uppercase sorts before lowercase")
print("key=str.lower     :", sorted(words, key=str.lower))
print("sorted a tuple    :", sorted((3, 1, 2)), "| [3, 1, 2].sort() returns:", [3, 1, 2].sort())
assert [c["id"] for c in by_score] == ["c2", "c3", "c1", "c4"]            # tie kept in original order
assert sorted(words) == ["Cherry", "apple", "banana"] and sorted(words, key=str.lower) == ["apple", "banana", "Cherry"]

## 4 · Alias, shallow copy, deep copy

| | How | New outer list? | New inner objects? |
|---|---|---|---|
| **Alias** | `b = a` | no | no |
| **Shallow copy** | `a.copy()`, `a[:]`, `list(a)`, `copy.copy(a)` | yes | **no** — the elements are shared |
| **Deep copy** | `copy.deepcopy(a)` | yes | yes, recursively |

A shallow copy is enough for a list of immutable things (strings, numbers). With **nested mutable** data — a list of dicts, like chat messages — the copy shares the inner dicts.

**Predict, then run:** a shared system-prompt template is copied for a new user, and that user's copy is edited. Which lists change?

In [ ]:
import copy

TEMPLATE = [{"role": "system", "content": "You are a concise assistant."}]

alias = TEMPLATE
shallow = TEMPLATE.copy()
deep = copy.deepcopy(TEMPLATE)

shallow.append({"role": "user", "content": "Hi"})       # outer list: only `shallow` grows
shallow[0]["content"] = "You are a pirate."             # inner dict: SHARED with TEMPLATE

print("TEMPLATE:", TEMPLATE)
print("alias   :", alias)
print("deep    :", deep)
print("len(TEMPLATE), len(shallow):", len(TEMPLATE), len(shallow))
assert TEMPLATE[0]["content"] == "You are a pirate."            # the bug
assert deep[0]["content"] == "You are a concise assistant."     # the fix
assert shallow is not TEMPLATE and shallow[0] is TEMPLATE[0]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_copies(cases):
    """One panel per (title, a, b): names -> outer lists -> inner lists. Amber = reachable from both a and b."""
    fig, axes = plt.subplots(1, len(cases), figsize=(11, 2.8))
    for ax, (title, a, b) in zip(axes, cases):
        outers = [a] if a is b else [a, b]
        outer_y = {id(a): 1.5} if a is b else {id(a): 2.3, id(b): 0.7}
        inners = []
        for outer in outers:
            for item in outer:
                if not any(item is seen for seen in inners):
                    inners.append(item)
        step = 2.4 / max(len(inners) - 1, 1)
        inner_y = {id(item): 2.7 - i * step for i, item in enumerate(inners)}

        def box(x, y, w, text, shared):
            ax.add_patch(FancyBboxPatch((x, y - 0.2), w, 0.4, boxstyle="round,pad=0.04",
                                        fc="#ffe8a3" if shared else "#e7f0ff", ec="#3b5bdb"))
            ax.text(x + w / 2, y, text, ha="center", va="center", fontsize=8, family="monospace")

        for name, obj, y in (("a", a, 2.3), ("b", b, 0.7)):
            ax.text(0.1, y, name, ha="center", va="center", fontsize=12, family="monospace")
            ax.annotate("", xy=(0.75, outer_y[id(obj)]), xytext=(0.3, y), arrowprops=dict(arrowstyle="->", lw=1.1))
        for outer in outers:
            box(0.8, outer_y[id(outer)], 1.0, "outer list", shared=a is b)
            for item in outer:
                ax.annotate("", xy=(2.35, inner_y[id(item)]), xytext=(1.85, outer_y[id(outer)]),
                            arrowprops=dict(arrowstyle="->", lw=1.0, color="#495057"))
        for item in inners:
            shared = any(item is x for x in a) and any(item is x for x in b)
            box(2.4, inner_y[id(item)], 1.3, repr(item), shared)
        ax.set_xlim(-0.1, 3.9); ax.set_ylim(0.0, 3.0); ax.axis("off"); ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

nested = [[1, 2], [3, 4]]
draw_copies([("alias: b = a", nested, nested),
             ("shallow: b = a.copy()", nested, nested.copy()),
             ("deep: b = copy.deepcopy(a)", nested, copy.deepcopy(nested))])

Amber means "one object reachable from both `a` and `b`" — a change made through one name shows up through the other. The alias shares everything, the shallow copy shares the inner lists, the deep copy shares nothing.

## 5 · The `[[0] * 3] * 3` trap

`*` on a list repeats the **references**; it does not copy the items. For a list of numbers that is fine (ints are immutable). For a list of lists you get three references to *one* inner list. The same trap appears as `dict.fromkeys(keys, [])` ([04](04_dicts_and_sets.ipynb)).

**Predict, then run:** after `grid[0][0] = 1`, what does `grid` look like?

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 1
print("grid   :", grid)
print("all three rows are one object?", grid[0] is grid[1] is grid[2])

grid_ok = [[0] * 3 for _ in range(3)]     # a new inner list for every row
grid_ok[0][0] = 1
print("grid_ok:", grid_ok)
assert grid == [[1, 0, 0]] * 3 and grid_ok == [[1, 0, 0], [0, 0, 0], [0, 0, 0]]

## 6 · Unpacking, including `*`

Any iterable can be unpacked into names. The counts must match, unless one name is starred: it collects "the rest" into a **list** (possibly empty). `_` is the convention for "I don't need this one".

In [ ]:
role, content = ("user", "What is RAG?")
first, *middle, last = [10, 20, 30, 40, 50]
head, *tail = "abc"
system_msg, *turns = ["sys", "u1", "a1", "u2"]
only, *nothing = ["solo"]
name, (lat, lon) = ("Paris", (48.86, 2.35))              # nested unpacking
_, score = ("doc-7", 0.93)
print(role, "|", first, middle, last, "|", head, tail, "|", system_msg, turns, "|", only, nothing, "|", name, lat, lon, "|", score)

try:
    x, y = [1, 2, 3]
except ValueError as err:
    print("ValueError:", err)
assert middle == [20, 30, 40] and tail == ["b", "c"] and turns == ["u1", "a1", "u2"] and nothing == []

## 7 · Tuples: fixed records, immutable — mostly

A tuple is an ordered, **immutable** sequence. Use it for a fixed-shape record where each position has a meaning — `(doc_id, score)`, `(lat, lon)`, several return values — and as a dict key. The **comma** makes the tuple, not the parentheses: `(1)` is an int, `(1,)` is a tuple.

Immutable means the tuple's **slots** cannot be rebound. If a slot holds a mutable object, that object can still change — and then the tuple is no longer hashable.

**Predict, then run:** `record[1] += ["faq"]` raises a `TypeError`. Does the list inside change anyway?

In [ ]:
print("type((1)):", type((1)).__name__, "| type((1,)):", type((1,)).__name__, "| 1, 2 ->", (1, 2))

record = ("doc-7", ["refund"])
try:
    record[0] = "doc-8"
except TypeError as err:
    print("TypeError:", err)

record[1].append("policy")                 # fine: we mutate the list, not the tuple
try:
    record[1] += ["faq"]                    # list.__iadd__ extends the list, THEN the slot assignment fails
except TypeError as err:
    print("TypeError:", err)
print("record:", record)

cache = {("gpt-4.1-mini", "What is RAG?"): "cached answer"}      # tuples of immutables are hashable
print("tuple as a dict key:", cache[("gpt-4.1-mini", "What is RAG?")])
try:
    hash(record)
except TypeError as err:
    print("hash(record) -> TypeError:", err)
assert record == ("doc-7", ["refund", "policy", "faq"])

### `namedtuple`: a tuple with field names

`collections.namedtuple`, or its typed cousin `typing.NamedTuple`, names each position while keeping all tuple behaviour — unpacking, indexing, hashing, small memory — and adds a readable `repr`. For records you need to change or validate, use a dataclass ([08](08_classes_oop_dataclasses.ipynb)).

In [ ]:
from collections import namedtuple
from typing import NamedTuple

ScoredDoc = namedtuple("ScoredDoc", ["doc_id", "score"])

class Usage(NamedTuple):
    prompt_tokens: int
    completion_tokens: int = 0

hit = ScoredDoc("doc-7", 0.93)
doc_id, doc_score = hit                        # still unpacks like a tuple
print(hit, "| hit.score:", hit.score, "| hit[0]:", hit[0])
print("_replace:", hit._replace(score=0.5), "| _asdict:", hit._asdict())
usage = Usage(120)
print(usage, "| sum(usage):", sum(usage))
assert hit == ("doc-7", 0.93) and usage.completion_tokens == 0 and sum(usage) == 120

## 8 · List vs tuple

| | `list` | `tuple` |
|---|---|---|
| Mutable | yes | no — the slots are fixed |
| Hashable (dict key, set member) | no | yes, if every item is hashable |
| Typical meaning | a **sequence** of similar things (tokens, messages, results) | a **record** of positions (id, score) |
| Methods | many (`append`, `sort`, …) | just `count` and `index` |
| Memory | may hold spare capacity so `append` is cheap | exactly sized, a little smaller |
| Safety | callers can change it | safe to share and to use as a default value |

In [ ]:
import sys

items = list(range(10))
grown = []
for i in range(10):
    grown.append(i)
print("getsizeof list(range(10)):", sys.getsizeof(items), "| tuple:", sys.getsizeof(tuple(items)),
      "| list built by 10 appends:", sys.getsizeof(grown), "(spare capacity)")
assert sys.getsizeof(tuple(items)) < sys.getsizeof(items) <= sys.getsizeof(grown)

## 9 · What operations cost — and why `append` is cheap

A list is a contiguous array of pointers. Indexing is arithmetic, so O(1). Inserting or removing at the front shifts every pointer after it, so O(n). `append` is **amortised O(1)**: when the array is full, CPython allocates a bigger one (about 1.125× the size, plus a few slots) and copies once; the occasional copies average out to a constant cost per append. You can watch the capacity jump:

| Operation | Cost |
|---|---|
| `lst[i]`, `lst[i] = x`, `len(lst)` | O(1) |
| `append(x)`, `pop()` | O(1) amortised |
| `insert(0, x)`, `pop(0)`, `insert(i, x)`, `del lst[i]` | O(n) |
| `x in lst`, `index`, `remove`, `count` | O(n) |
| `lst[a:b]` (makes a copy) | O(b − a) |
| `sort()` | O(n log n) |

In [ ]:
lengths, sizes = [], []
lst = []
for i in range(80):
    lengths.append(len(lst))
    sizes.append(sys.getsizeof(lst))
    lst.append(i)
reallocations = sum(1 for before, after in zip(sizes, sizes[1:]) if after > before)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.step(lengths, sizes, where="post", color="#3b5bdb")
ax.set_title(f"80 appends, only {reallocations} reallocations: capacity grows in jumps")
ax.set_xlabel("len(list) before the append"); ax.set_ylabel("sys.getsizeof (bytes)")
plt.show()
assert reallocations < 15

Front-insertion is where lists hurt. `collections.deque` (a double-ended queue) has O(1) appends and pops at **both** ends — the right structure for a FIFO queue (a BFS frontier, a work queue) or a sliding window of recent messages (`deque(maxlen=N)` drops the oldest item automatically). The trade-off: indexing into the middle of a deque is O(n).

In [ ]:
import timeit
from collections import deque

def build_append(n):
    out = []
    for i in range(n):
        out.append(i)
    return out

def build_insert_front(n):
    out = []
    for i in range(n):
        out.insert(0, i)
    return out

def build_deque_appendleft(n):
    out = deque()
    for i in range(n):
        out.appendleft(i)
    return out

ns = [2_000, 4_000, 8_000, 16_000]
funcs = [build_append, build_insert_front, build_deque_appendleft]
ms = {f.__name__: [min(timeit.repeat(lambda: f(n), number=1, repeat=3)) * 1000 for n in ns] for f in funcs}

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(ns, ms["build_insert_front"], marker="o", color="#e8590c", label="list.insert(0, x)")
ax.plot(ns, ms["build_deque_appendleft"], marker="s", color="#2f9e44", label="deque.appendleft(x)")
ax.plot(ns, ms["build_append"], marker="^", ls="--", color="#1c7ed6", label="list.append(x)  (overlaps deque)")
ax.set_title("Adding n items: list.insert(0, x) shifts the whole list every time")
ax.set_xlabel("n items"); ax.set_ylabel("total time (ms)"); ax.legend()
plt.show()
print({name: f"{times[-1]:.1f} ms" for name, times in ms.items()}, "at n =", ns[-1])
assert list(build_deque_appendleft(5)) == build_insert_front(5) == [4, 3, 2, 1, 0]
assert ms["build_insert_front"][-1] > 5 * ms["build_append"][-1]

In [ ]:
queue = deque(["task-1", "task-2"])
queue.append("task-3")                            # enqueue on the right
print("popleft():", queue.popleft(), "| remaining:", list(queue))   # dequeue on the left, O(1)

recent = deque(maxlen=3)                           # a sliding window of the last 3 turns
for turn in ["u1", "a1", "u2", "a2", "u3"]:
    recent.append(turn)
print("last 3 turns:", list(recent))
assert list(recent) == ["u2", "a2", "u3"]

## 10 · `enumerate`, `zip`, `reversed`

Loop over the items directly and let these helpers supply what you would otherwise index for:

- `enumerate(seq, start=1)` gives `(index, item)` pairs.
- `zip(a, b)` gives pairs and **stops at the shortest input — silently**. `zip(..., strict=True)` (3.10+) raises instead; use it whenever the lengths must match, such as questions and answers in an eval set.
- `reversed(seq)` is a reverse iterator — no copy, unlike `seq[::-1]`.
- `zip(*pairs)` "unzips" a list of pairs.

**Predict, then run:** how many pairs does the plain `zip` produce?

In [ ]:
questions = ["What is RAG?", "What is LoRA?", "What is BM25?"]
answers = ["Retrieval-augmented generation", "Low-rank adaptation"]        # one answer missing!

for i, q in enumerate(questions, start=1):
    print(f"Q{i}: {q}")
print("zip pairs:", len(list(zip(questions, answers))))
try:
    list(zip(questions, answers, strict=True))
except ValueError as err:
    print("strict=True -> ValueError:", err)

print("reversed:", list(reversed(questions)))
ids, scores = zip(*[("d1", 0.9), ("d2", 0.7)])
print("unzipped:", ids, scores)
assert len(list(zip(questions, answers))) == 2 and ids == ("d1", "d2")

## 11 · Practical: a chat history is a list of dicts

Every chat API takes `messages`: a list of `{"role": ..., "content": ...}` dicts, oldest first. The common operations are all list operations. One rule from section 4 matters here: **don't mutate the caller's list** unless that is the documented contract — return a new list.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful support assistant."},
    {"role": "user", "content": "Hi, I need a refund."},
    {"role": "assistant", "content": "Sure - what is your order number?"},
    {"role": "user", "content": "It's 4417."},
    {"role": "assistant", "content": "Thanks, checking order 4417."},
    {"role": "user", "content": "How long will it take?"},
]

last_user = next(m for m in reversed(messages) if m["role"] == "user")
print("last user message:", last_user["content"])
print("roles            :", [m["role"] for m in messages])
print("user turns       :", sum(m["role"] == "user" for m in messages))
print("words in history :", sum(len(m["content"].split()) for m in messages), "(a rough stand-in for tokens)")
assert last_user["content"] == "How long will it take?"

In [ ]:
def trim_history(messages, max_turns):
    """Keep the system message(s) plus the last `max_turns` other messages. Returns a NEW list."""
    system = [m for m in messages if m["role"] == "system"]
    others = [m for m in messages if m["role"] != "system"]
    return system + others[-max_turns:]

def fit_budget(messages, max_words):
    """Drop the oldest non-system messages until the history fits a word budget."""
    kept = list(messages)                      # shallow copy: we remove items but never edit them
    while sum(len(m["content"].split()) for m in kept) > max_words and len(kept) > 2:
        kept.pop(1)                            # index 0 is the system message
    return kept

short = trim_history(messages, max_turns=2)
fitted = fit_budget(messages, max_words=20)
print("trim_history:", [m["role"] for m in short])
print("fit_budget  :", [m["content"] for m in fitted])
print("the original still has", len(messages), "messages")
assert [m["role"] for m in short] == ["system", "assistant", "user"] and len(messages) == 6
assert sum(len(m["content"].split()) for m in fitted) <= 20 and fitted[0]["role"] == "system"

Real systems count tokens with the model's tokenizer instead of words, and often summarise the dropped turns rather than discarding them — see [conversation memory and context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb).

## Try it yourself

**1.** Three documents each produced a list of `(chunk_id, score)` hits. Combine them into one list and return the top 3 by score.

In [ ]:
# Solution — try it yourself first, then run this
hits_per_doc = [[("faq-1", 0.72), ("faq-2", 0.31)], [("policy-1", 0.91)], [("blog-1", 0.55), ("blog-2", 0.88)]]
all_hits = []
for hits in hits_per_doc:
    all_hits.extend(hits)                       # extend, not append: we want a flat list
top3 = sorted(all_hits, key=lambda h: h[1], reverse=True)[:3]
print(top3)
assert [cid for cid, _ in top3] == ["policy-1", "blog-2", "faq-1"]

**2.** Rotate a list left by `k` positions using slicing, and check your answer against `deque.rotate` (which rotates *right* for positive numbers).

In [ ]:
# Solution — try it yourself first, then run this
def rotate_left(items, k):
    if not items:
        return []
    k %= len(items)                             # handles k > len(items) and negative k
    return items[k:] + items[:k]

d = deque([1, 2, 3, 4, 5])
d.rotate(-2)                                    # negative = rotate left
print(rotate_left([1, 2, 3, 4, 5], 2), list(d))
assert rotate_left([1, 2, 3, 4, 5], 2) == list(d) == [3, 4, 5, 1, 2]
assert rotate_left([1, 2, 3], 7) == [2, 3, 1] and rotate_left([], 3) == []

**3.** Sort `(name, score)` pairs by score **descending**, breaking ties by name **ascending** — once with a single tuple key, once with two stable sorts. Show both give the same answer.

In [ ]:
# Solution — try it yourself first, then run this
pairs = [("carol", 0.8), ("alice", 0.9), ("bob", 0.8), ("dave", 0.9)]
one_pass = sorted(pairs, key=lambda p: (-p[1], p[0]))
two_pass = sorted(sorted(pairs, key=lambda p: p[0]), key=lambda p: p[1], reverse=True)   # secondary key first
print(one_pass)
assert one_pass == two_pass == [("alice", 0.9), ("dave", 0.9), ("bob", 0.8), ("carol", 0.8)]

**4.** Some chat APIs reject two consecutive messages with the same role. Write `merge_consecutive(messages)` that joins such runs with a newline — **without modifying** the input list or its dicts.

In [ ]:
# Solution — try it yourself first, then run this
def merge_consecutive(messages):
    merged = []
    for m in messages:
        if merged and merged[-1]["role"] == m["role"]:
            merged[-1] = {**merged[-1], "content": merged[-1]["content"] + "\n" + m["content"]}   # new dict
        else:
            merged.append(dict(m))                                                                # a copy
    return merged

raw = [{"role": "user", "content": "Hi"}, {"role": "user", "content": "Are you there?"},
       {"role": "assistant", "content": "Yes!"}]
snapshot = copy.deepcopy(raw)
out = merge_consecutive(raw)
print(out)
assert [m["role"] for m in out] == ["user", "assistant"] and out[0]["content"] == "Hi\nAre you there?"
assert raw == snapshot                        # the input was not touched

## Say it in an interview

- **List vs tuple (30 s):** "A list is a mutable sequence of similar items; a tuple is an immutable, fixed-shape record where each position means something. Tuples are hashable if their contents are, so they can be dict keys — for example a cache key of `(model, prompt)`. But a tuple holding a list can still have that list change."
- **Complexity:** indexing, `append`, `pop()` are O(1) (append amortised thanks to over-allocation); `insert(0)`, `pop(0)`, `in`, `remove` are O(n); `sort` is O(n log n). For a queue use `collections.deque` — O(1) at both ends.
- **Copying:** `b = a` aliases; `a.copy()` / `a[:]` is shallow — new outer list, shared elements; `copy.deepcopy` copies recursively. Shallow copies of nested mutable data (lists of dicts, like chat messages) are the classic bug.
- **Sorting:** `sorted()` returns a new list from any iterable, `.sort()` is in place and returns `None`; Timsort is stable, so multi-key sorts work with a tuple key or with successive sorts.
- **Traps:** `[[0] * 3] * 3` repeats one inner list; `x = lst.append(...)` is `None`; `zip` silently truncates (use `strict=True`); `(1)` is not a tuple; `t[0] += [x]` both mutates the list *and* raises.
- **Follow-up they ask:** "How does append stay O(1) when the array fills up?" — geometric over-allocation: each resize grows capacity by a constant factor, so total copying is O(n) over n appends.

## Next

- [04 · Dicts and sets](04_dicts_and_sets.ipynb) — hashing, counting, grouping and de-duplicating.
- [Arrays and two pointers](../03_dsa/02_arrays_and_two_pointers.ipynb) and [stacks and queues](../03_dsa/04_stacks_and_queues.ipynb) — the DSA side of lists and deques.
- [Conversation memory and context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb) — managing `messages` for real.
- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb).
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 4: collections) · the [dsa_basics course](../../dsa_basics/index.html).